In [1]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                           get_linear_schedule_with_warmup)
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, f1_score
import matplotlib.pyplot as plt
import os, time

c:\Users\Buwaneka Fernando\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ── Check GPU availability ────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU found — training will be slow. Use Google Colab for GPU.")

# ── Load data ─────────────────────────────────────────────
train = pd.read_csv("../data/final/train.csv")
val   = pd.read_csv("../data/final/val.csv")
test  = pd.read_csv("../data/final/test.csv")

print(f"Train: {len(train)} | Val: {len(val)} | Test: {len(test)}")

Using device: cpu
No GPU found — training will be slow. Use Google Colab for GPU.
Train: 3624 | Val: 777 | Test: 777


In [3]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

CUDA available: False


In [4]:
# DATASET CLASS — converts your CSV rows into PyTorch tensors

class CognitiveDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts      = texts.tolist()
        self.labels     = labels.tolist()
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }


In [5]:
#TRAINING FUNCTION — reusable for both DistilBERT & RoBERTa
def train_transformer(model_name, save_dir, num_epochs=5,
                      batch_size=16, lr=2e-5, max_length=256):
    """
    Fine-tune any HuggingFace classifier model.
    Returns: trained model, tokenizer, training history
    """
    print(f"\n{'='*55}")
    print(f"  TRAINING: {model_name}")
    print(f"  Epochs: {num_epochs} | Batch: {batch_size} | LR: {lr}")
    print(f"{'='*55}")

    # ── Load tokenizer and model ─────────────────────────
    print("Loading tokenizer and model...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model     = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2,           # Binary: System 1 vs System 2
        ignore_mismatched_sizes=True
    )
    model = model.to(device)

    # ── Create datasets and loaders ──────────────────────
    train_ds = CognitiveDataset(train['input_text'], train['cognitive_label'],
                                tokenizer, max_length)
    val_ds   = CognitiveDataset(val['input_text'],   val['cognitive_label'],
                                tokenizer, max_length)

    train_loader = DataLoader(train_ds, batch_size=batch_size,
                              shuffle=True,  num_workers=0)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size,
                              shuffle=False, num_workers=0)

    # ── Optimizer and scheduler ──────────────────────────
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=0.01)

    total_steps   = len(train_loader) * num_epochs
    warmup_steps  = total_steps // 10   # 10% warmup

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

    # ── Training loop ────────────────────────────────────
    history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_f1': []}
    best_val_f1   = 0
    best_epoch    = 0

    for epoch in range(num_epochs):
        start_time = time.time()

        # ── Training phase ───────────────────────────────
        model.train()
        train_losses = []

        for batch_idx, batch in enumerate(train_loader):
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['label'].to(device)

            optimizer.zero_grad()

            outputs = model(input_ids=input_ids,
                            attention_mask=attention_mask,
                            labels=labels)

            loss = outputs.loss
            loss.backward()

            # Gradient clipping — prevents exploding gradients
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()
            scheduler.step()

            train_losses.append(loss.item())

            # Print progress every 50 batches
            if (batch_idx + 1) % 50 == 0:
                print(f"  Epoch {epoch+1} | Batch {batch_idx+1}/{len(train_loader)} "
                      f"| Loss: {loss.item():.4f}", end='\r')

        avg_train_loss = np.mean(train_losses)

        # ── Validation phase ─────────────────────────────
        model.eval()
        val_losses  = []
        all_preds   = []
        all_labels  = []

        with torch.no_grad():
            for batch in val_loader:
                input_ids      = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels         = batch['label'].to(device)

                outputs = model(input_ids=input_ids,
                                attention_mask=attention_mask,
                                labels=labels)

                val_losses.append(outputs.loss.item())

                preds = torch.argmax(outputs.logits, dim=1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        avg_val_loss = np.mean(val_losses)
        val_acc      = accuracy_score(all_labels, all_preds)
        val_f1       = f1_score(all_labels, all_preds, average='weighted')
        elapsed      = time.time() - start_time

        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['val_acc'].append(val_acc)
        history['val_f1'].append(val_f1)

        print(f"\n  Epoch {epoch+1}/{num_epochs} | "
              f"Train Loss: {avg_train_loss:.4f} | "
              f"Val Loss: {avg_val_loss:.4f} | "
              f"Val Acc: {val_acc:.4f} | "
              f"Val F1: {val_f1:.4f} | "
              f"Time: {elapsed:.0f}s")

        # ── Save best model checkpoint ───────────────────
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_epoch  = epoch + 1
            os.makedirs(save_dir, exist_ok=True)
            model.save_pretrained(save_dir)
            tokenizer.save_pretrained(save_dir)
            print(f"  ✓ Best model saved (Val F1: {val_f1:.4f})")

    print(f"\n  Training complete.")
    print(f"  Best Val F1: {best_val_f1:.4f} at epoch {best_epoch}")

    return model, tokenizer, history

In [6]:
# TRAIN MODEL 1 — DistilBERT 
distilbert_model, distilbert_tokenizer, distilbert_history = train_transformer(
    model_name = "distilbert-base-uncased",
    save_dir   = "../models/distilbert_checkpoint",
    num_epochs = 5,
    batch_size = 16,
    lr         = 2e-5
)


  TRAINING: distilbert-base-uncased
  Epochs: 5 | Batch: 16 | LR: 2e-05
Loading tokenizer and model...


c:\Users\Buwaneka Fernando\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Buwaneka Fernando\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is n

KeyboardInterrupt: 

In [ ]:
# TRAIN MODEL 2 — RoBERTa 
roberta_model, roberta_tokenizer, roberta_history = train_transformer(
    model_name = "roberta-base",
    save_dir   = "../models/roberta_checkpoint",
    num_epochs = 5,
    batch_size = 16,
    lr         = 1e-5   # RoBERTa needs slightly lower LR than DistilBERT
)

In [ ]:
# PLOT TRAINING CURVES — goes directly into your paper
# ════════════════════════════════════════════════════════
def plot_training_history(history, model_name):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Loss curve
    axes[0].plot(history['train_loss'], label='Train Loss', marker='o')
    axes[0].plot(history['val_loss'],   label='Val Loss',   marker='o')
    axes[0].set_title(f"{model_name} — Loss Curve")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Accuracy & F1 curve
    axes[1].plot(history['val_acc'], label='Val Accuracy', marker='o')
    axes[1].plot(history['val_f1'],  label='Val F1',       marker='s')
    axes[1].set_title(f"{model_name} — Val Metrics")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Score")
    axes[1].set_ylim(0.5, 1.0)
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    fname = model_name.lower().replace(' ', '_').replace('-', '_')
    plt.savefig(f"../outputs/figures/training_curve_{fname}.png", dpi=150)
    plt.show()
    print(f"Saved → ../outputs/figures/training_curve_{fname}.png")

plot_training_history(distilbert_history, "DistilBERT")
plot_training_history(roberta_history,    "RoBERTa")